## Vis for labels: OrigImg, GT, Preds
### This part is for YYF_30Case data and WSSS_Unet model

In [25]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os
from os.path import join

In [ ]:
data_root_dir = "../data/YYF_30Case"
pred_root_dir = "../experiments/wsss_unet/results/YYF_30Case"
pred_full_root_dir = pred_root_dir.replace("wsss_unet", "full_supervised_unet")
save_dir = join("../experiments/result_vis/YYF_30Case/labels_vis/")
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
# preprocessed images
img_dir = join(data_root_dir, "preprocessed_size256")
img_cropped_dir = join(data_root_dir, "preprocessed_size256_cropped")
# our prediction
pred_full_dir = join(pred_full_root_dir, "preprocessed_size256/pred_mask")
pred_full_cropped_dir = join(pred_full_root_dir, "preprocessed_size256_cropped/pred_mask")
pred_dir = join(pred_root_dir, "preprocessed_size256/pred_mask")
pred_cropped_dir = join(pred_root_dir, "preprocessed_size256_cropped/pred_mask")
# labels from sheng zhang and YYF
labels_1_dir = join(data_root_dir, "labels_1_imgs")
labels_2_dir = join(data_root_dir, "labels_2_imgs")

all_dirs = [
    img_dir,
    pred_full_dir,
    pred_dir,
    labels_1_dir,
    labels_2_dir,
    img_cropped_dir,
    pred_full_cropped_dir,
    pred_cropped_dir,
    
]
# check if all directories exist
for d in all_dirs:
    if not os.path.exists(d):
        print(f"Directory {d} does not exist.")
        exit(1)

In [ ]:
from typing import List
def get_img_list(img_dir: str, case_name: str):
    """
    Get the list of images in the specified directory.
    Args:
        img_dir (str): Directory containing the images.
        case_index (int): Index of the case to process.
    Returns:
        dict: Dictionary containing the image names and paths.
    """
    # Get the list of cases
    cases_list = os.listdir(img_dir)
    cases_list = [f for f in cases_list if '.' not in f]
    
    if isinstance(case_name, str):
        case_id = case_name if case_name in cases_list else case_name.rsplit('_', 1)[0]# for case_id as '1', 'AIIB23_146'
    elif isinstance(case_name, int):
        case_id = cases_list[case_name]
    else:
        raise ValueError("case_name should be either str or int")

    imgs = {}
    try:
        imgs['name'] = sorted(os.listdir(join(img_dir, case_id)))
        imgs['path'] = [join(img_dir, case_id, f) for f in imgs['name']]
        imgs['case_id'] = case_id
    except FileNotFoundError:
        case_id = case_id + '_0000'
        imgs['name'] = sorted(os.listdir(join(img_dir, case_id)))
        imgs['path'] = [join(img_dir, case_id, f) for f in imgs['name']]
        imgs['case_id'] = case_id
    # print("img name", imgs['name'][:3])
    # print("img path", imgs['path'][:3])
    
    return imgs
    
def load_img(img_path_list: List[str]) -> np.ndarray:
    """
    Load images from the specified paths.
    Args:
        img_path_list (list[str]): List of image paths.
    Returns:
        np.ndarray: Array of loaded images.
    """
    img_list = []
    for img_path in img_path_list:
        img = Image.open(img_path)
        img = np.array(img)
        img_list.append(img)
    img = np.stack(img_list, axis=0)
    return img

def vis_imgs(save_dir, img_list: List[np.ndarray], middle_slice_id: int = 0, case_id: str = "case_0", show_cropped: bool = False, show: bool = False) -> None:
    """
    Visualize images in a grid.

    Args: 
        save_dir (str): Directory to save the visualizations.
        img_list (list[np.ndarray]): List of image arrays to visualize.
        middle_slice_id (int): Starting slice index for visualization.
        case_id (str): Identifier for the case being visualized.
        show_cropped (bool): Whether to include cropped images in the visualization.
        show (bool): Whether to display the plot interactively.
    """
    all_titles = ["orig", "full_super", "wsss_unet", "Labels_1", "Labels_2", "orig_cropped", "full_super_cropped", "wsss_unet_cropped"]
    if show_cropped:
        all_titles = [strs for strs in all_titles if "cropped" in strs.lower()]
        all_imgs = img_list[-len(all_titles):]
    else:
        all_titles = [strs for strs in all_titles if "cropped" not in strs.lower()]
        all_imgs = img_list[:len(all_titles)]  # Exclude cropped images

    fig, axes = plt.subplots(len(all_imgs), 8, figsize=(20, len(all_imgs) * 3))
    for i in range(8):
        for j, imgs in enumerate(all_imgs):
            axes[j, i].imshow(imgs[i], cmap='gray')
            axes[j, i].set_title(all_titles[j] + f" {middle_slice_id + i}")
            axes[j, i].axis('off')

    plt.suptitle(f"Case {case_id} - Slices {middle_slice_id} to {middle_slice_id + 8}", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.99])  # Adjust layout to fit the suptitle

    img_name = f"case_{case_id}_slice_{middle_slice_id}_cropped.png" if show_cropped else f"case_{case_id}_slice_{middle_slice_id}.png"
    plt.savefig(join(save_dir, img_name), dpi=300)
    print(f"Saved figure for case {case_id} at slice {middle_slice_id}.")

    plt.show() if show else plt.close()


In [ ]:
cases_list = os.listdir(labels_2_dir)
cases_list = [f for f in cases_list if '.' not in f]
print(f"cases name {cases_list[:5]}\n")
# case_name = '3_0000'
for case_name in cases_list:

    all_img_list = [get_img_list(img_dir, case_name = case_name) for img_dir in all_dirs]
    img_list, pred_full_list, pred_list, labels_1_list, labels_2_list, img_cropped_list, pred_full_cropped_list, pred_cropped_list = all_img_list
    case_id = img_list['case_id']

    middle_slice_idx = len(img_list['name']) // 2
    for middle_slice_id in range(middle_slice_idx - 100, middle_slice_idx + 101, 50):
        slice_name = img_list['name'][middle_slice_id]
        print(f"processing case_id: {case_id}; slices: {middle_slice_id}; slice_name: {slice_name}")

        # get the middle slice name of the cropped image, as the cropped image doest have top and bottom k imgs which does not have lung 
        try:
            converted_id = img_cropped_list['name'].index(slice_name)
        except:
            converted_id = img_cropped_list['name'].index(slice_name.replace("_0000", ""))


        try:
            imgs_np, pred_full_np, pred_np, labels_1_np, labels_2_np = [load_img(list_name['path'][middle_slice_id: middle_slice_id+8]) for list_name in all_img_list[:5]]    
            imgs_cropped_np, pred_full_cropped_np, pred_cropped_np = [load_img(list_name['path'][converted_id: converted_id+8]) for list_name in all_img_list[5:]]
        except FileNotFoundError:
            print(f"File not found for case {case_id} at slice {middle_slice_id}.")
            continue

        all_imgs = [imgs_np, pred_full_np, pred_np, labels_1_np, labels_2_np, imgs_cropped_np, pred_full_cropped_np, pred_cropped_np]
        vis_imgs(save_dir, all_imgs, middle_slice_id, case_id, show_cropped=False)
        vis_imgs(save_dir, all_imgs, middle_slice_id, case_id, show_cropped=True)
        # break
